In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import os
import gc

BASE_PATH = "/content/drive/MyDrive/hand2font_project"
TRAIN_CSV = os.path.join(BASE_PATH, "emnist-byclass-train.csv")
TEST_CSV  = os.path.join(BASE_PATH, "emnist-byclass-test.csv")
MODEL_SAVE_PATH = os.path.join(BASE_PATH, "efficientNetB1.pth")

#  מילון איחוד אותיות
def get_collapsed_label(original_label):
    mapping = {
        38: 12, 45: 19, 46: 20, 47: 21, 48: 22, 50: 24,
        51: 25, 54: 28, 56: 30, 57: 31, 58: 32, 59: 33, 61: 35
    }
    return mapping.get(original_label, original_label)

# אינדקסים רציפים
ALL_LABELS   = sorted(list(set([get_collapsed_label(i) for i in range(10, 62)])))
LABEL_TO_IDX = {label: i for i, label in enumerate(ALL_LABELS)}
NUM_CLASSES  = len(ALL_LABELS)   # = 39

# ניקוי והכנת תמונה ל-EfficientNet
def prepare_for_efficientnet_logic(raw_char_img, target_size=240):
    img_float = raw_char_img.astype(np.float32)
    min_val, max_val = 30, 200
    img_normalized = np.clip((img_float - min_val) * (255.0 / (max_val - min_val)), 0, 255).astype(np.uint8)
    h, w = img_normalized.shape[:2]
    scale = (target_size * 0.75) / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img_normalized, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    final = np.zeros((target_size, target_size), dtype=np.uint8) # יצירת קנבס
    x_off = (target_size - new_w) // 2
    y_off = (target_size - new_h) // 2
    final[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return final

class EMNISTCollapsedDataset(Dataset):
    def __init__(self, csv_path, target_size=240, nrows=None, is_training=True):

        df = pd.read_csv(csv_path, header=None, nrows=nrows)

        # סינון אותיות (לייבלים 10-61) ואיחוד
        df = df[df.iloc[:, 0] >= 10].copy()
        df.iloc[:, 0] = df.iloc[:, 0].apply(get_collapsed_label).map(LABEL_TO_IDX)

        self.data = df.reset_index(drop=True)
        del df
        gc.collect()

        self.target_size = target_size
        self.is_training = is_training

        if is_training:
            # אוגמנטציה זהירה
            self.transform = transforms.Compose([
                transforms.RandomRotation(10),
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
                transforms.RandomApply([
                    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 0.5))
                ], p=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx].values
        label = int(row[0])
        pixels = row[1:].astype(np.uint8).reshape(28, 28)
        img = np.transpose(pixels)
        img_cleaned = prepare_for_efficientnet_logic(img, self.target_size)
        img_pil = Image.fromarray(img_cleaned).convert('RGB')
        return self.transform(img_pil), label

In [ ]:
def run_training_initial():
    device = torch.device("cuda")

    train_loader = DataLoader(EMNISTCollapsedDataset(TRAIN_CSV, nrows=300000, is_training=True),
                              batch_size=64, shuffle=True, num_workers=2)
    test_loader  = DataLoader(EMNISTCollapsedDataset(TEST_CSV, nrows=30000, is_training=False),
                              batch_size=64, num_workers=2)

    # טעינת B1 עם משקולות ImageNet
    model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

    # אם עצרנו באמצע - ממשיכים מהמודל השמור
    if os.path.exists(MODEL_SAVE_PATH):
        try:
            model.load_state_dict(torch.load(MODEL_SAVE_PATH))
            print("--- המודל נטען מהדרייב. ממשיך מהנקודה האחרונה... ---")
        except:
            print("--- מתחיל אימון חדש (v2) ---")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        model.train()
        print(f"\n--- Epoch {epoch+1} ---")
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            if i % 200 == 0:
                print(f"Step {i}, Loss: {loss.item():.4f}")

        # בדיקת דיוק
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, pred = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (pred == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"⭐ Accuracy (39 classes): {accuracy:.2f}%")
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"המודל נשמר: {MODEL_SAVE_PATH}")

run_training_initial()

In [ ]:
def run_finetuning():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(EMNISTCollapsedDataset(TRAIN_CSV, nrows=300000, is_training=True),
                              batch_size=64, shuffle=True)
    test_loader  = DataLoader(EMNISTCollapsedDataset(TEST_CSV, nrows=30000, is_training=False),
                              batch_size=64)

    best_acc = 93.30

    # טוענים את המודל ששמרנו בשלב א'
    if os.path.exists(MODEL_SAVE_PATH):
        print("🔎 מצאתי מודל שמור! טוען...")
        model = models.efficientnet_b1(weights=None)   # שלד בלבד - המשקולות מגיעות מהקובץ השמור
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        model.load_state_dict(torch.load(MODEL_SAVE_PATH))
        model = model.to(device)
        print("✅ המודל נטען. ממשיכים בשיפור.")
    else:
        print("❌ לא נמצא מודל שמור. טוען משקולות בסיס של ImageNet...")
        model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.00001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(10):
        model.train()
        print(f"\n--- Epoch {epoch+1} ---")

        # מונים דיוק אימון תוך כדי האימון
        train_correct, train_total = 0, 0

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # צבירת דיוק אימון מתוך אותו forward
            _, train_pred = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (train_pred == labels).sum().item()

            if i % 200 == 0:
                print(f"Step {i}, Loss: {loss.item():.4f}")

        train_accuracy = 100 * train_correct / train_total

        # דיוק בדיקה (כמו קודם)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, pred = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (pred == labels).sum().item()

        accuracy = 100 * correct / total

        # מדפיסים את שניהם זה לצד זה
        print(f"📚 דיוק אימון: {train_accuracy:.2f}%  |  ⭐ דיוק בדיקה: {accuracy:.2f}%")

        if accuracy > best_acc:
            best_acc = accuracy
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"🏆 שיא חדש נשמר: {best_acc:.2f}%")

        if accuracy >= 95.0:
            break
run_finetuning()